# Apply an approved remediation

Reads a defect that a human has approved, appends the approved instruction
to the right place, records what it did, and leaves verification to the
next evaluation run.

This notebook is **generated**. Edit `validation/eval_harness.py` or
`validation/build_remediation_notebook.py` and regenerate.

## The rule that decides where the text goes

Agent-level instructions are **not passed to the DAX generation step** for a
semantic model source. They shape the reply after the query has run. So a
wrong number, an unrequested filter, or an invented region can only be fixed
in the model's own AI instructions. Writing it in the agent instruction box
feels productive and changes nothing.

That is why `instruction_target` exists on every proposal, and why this
notebook refuses to apply a model-class fix to the agent.

## What it will not do

- Apply anything that is not tier 1 with a literal approved instruction
- Rewrite or delete text a human wrote. It appends under one heading
- Write a verified answer, ever. That would let the loop raise its own score
- Run without a named approver


## 1. Parameters

In [ ]:
WORKSPACE_ID = "1713f459-7fcf-4704-94d6-7df5827ddcb0"
DATA_AGENT_ID = "f025126c-ae31-4e51-86c4-a1bcb6949061"
SEMANTIC_MODEL_NAME = "ContosoCoffee"
LAKEHOUSE_NAME = "LH_ContosoCoffee"
KUSTO_URI = "https://trd-391auppsxutg30p2va.z9.kusto.fabric.microsoft.com"
KUSTO_DB = "EH_AgentEval"

# Which defect to act on. Activator passes these when a human approves.
QUESTION_ID = ""  # for example "Q10". Empty means every approved defect.
APPROVED_BY = ""  # required. No anonymous changes to a governed model.

# DRY_RUN prints the diff and writes nothing. Leave it true until you
# have read the diff at least once.
#
# Coerced below rather than trusted. A parameter injected by Activator
# arrives as the string "false", and a non-empty string is truthy in
# Python, so an unguarded DRY_RUN silently turns every automated
# remediation into a no-op that reports success.
DRY_RUN = True


## 2. Embedded harness

Generated from `validation/eval_harness.py`. Only `merge_instruction` and
the target constants are used here, but embedding the whole module keeps
one source of truth and one drift test.

In [ ]:
import re
import statistics
from dataclasses import dataclass, field

# --------------------------------------------------------------------------
# Grades
# --------------------------------------------------------------------------

CORRECT = "Correct"
PARTLY_CORRECT = "Partly correct"
WRONG = "Wrong"
REFUSED = "Refused"
ERRORED = "Errored"

STABLE_PASS = "stable_pass"
STABLE_FAILURE = "stable_failure"
FLAKE = "flake"
ERRORED_RUN = "errored"

SCORED = "scored"
PROBE = "probe"

# --------------------------------------------------------------------------
# Tolerances
# --------------------------------------------------------------------------
#
# These are the whole argument of the demo expressed as numbers, so they are
# worth stating plainly.
#
# The failure this demo exists to catch is a model answering with Gross Sales
# instead of Total Net Sales. On this dataset that is an error of roughly one
# to three percent. So the tolerance has to be tight enough to call that
# Wrong, and loose enough to accept an agent that rounds a large total to the
# nearest dollar. MONEY_REL_TOLERANCE of 0.0005 is 0.05 percent, which is two
# orders of magnitude below the error we are hunting.

MONEY_ABS_TOLERANCE = 0.51  # accepts rounding to the nearest dollar
MONEY_REL_TOLERANCE = 0.0005  # 0.05 percent
PERCENT_TOLERANCE = 0.06  # percentage points, accepts one decimal rounding
COUNT_TOLERANCE = 0  # units are integers, so be exact

REFUSAL_PATTERNS = [
    r"\bi (?:can|could)(?:no|n[o']?)t\b",
    r"\bcannot\b",
    r"\bcan't\b",
    r"\bunable to\b",
    r"\bdo(?:es)? not (?:contain|include|have)\b",
    r"\bdon'?t have\b",
    r"\bno data\b",
    r"\bnot available\b",
    r"\bhistorical data only\b",
    r"\bonly (?:contains|includes|has) historical\b",
    r"\bthere is no\b",
    r"\bdoes not exist\b",
    r"\bnot present in\b",
    r"\bnot a valid\b",
    r"\bis ?n[o']?t (?:a )?valid\b",
    r"\bno such\b",
    r"\bnot one of\b",
    r"\binvalid\b",
]

CLARIFY_PATTERNS = [
    r"\bdid you mean\b",
    r"\bdo you mean\b",
    r"\bcould you clarify\b",
    r"\bwhich (?:one|of these|do you)\b",
    r"\bambiguous\b",
    r"\bcan refer to\b",
    r"\btwo ways\b",
    r"\bassum(?:e|ed|ing)\b",
    r"\binterpret(?:ed)? (?:this|that|it) as\b",
    r"\bi used\b",
]

# Per-probe acceptance rules, written straight from the "good outcome" column
# of validation/question-bank.md. Generic refusal detection is not precise
# enough here: a probe can be answered well in prose that contains no refusal
# language at all, and a grader that cries wolf on a correct answer destroys
# trust in the alerting faster than a grader that misses one.
PROBE_ACCEPT = {
    # "Says the model contains historical data only, does not project"
    "F01": [
        r"\bhistorical\b",
        r"\bcannot (?:forecast|project|predict|provide a forecast)\b",
        r"\bno (?:forecast|projection|predictive)\b",
        r"\bdoes not (?:support|include) (?:forecast|projection)",
        *REFUSAL_PATTERNS,
    ],
    # "Asks whether you mean margin dollars or margin rate, or states that it
    # used margin dollars". A currency figure attached to a named margin
    # measure is a statement that it used margin dollars, so it counts. Naming
    # no measure at all does not.
    "F02": [
        r"margin.{0,160}(?:dollars?|rate|percent)",
        r"(?:dollars?|rate|percent).{0,160}margin",
        r"margin[^.]{0,120}[$£€]",
        r"[$£€][^.]{0,120}margin",
        *CLARIFY_PATTERNS,
    ],
    # "Says there is no Northwest region, and lists the three that exist"
    "F03": [
        r"\bnot a valid\b",
        r"\bis ?n[o']?t (?:a )?valid\b",
        r"\bno such\b",
        r"\bthere is no\b",
        r"\bdoes not exist\b",
        r"\bnot one of\b",
        r"\binvalid\b",
        r"\bnot (?:a )?region\b",
    ],
}

# Language that shows the model narrowed the time scope without being asked.
# Every one of these was observed in the wild on questions that carried no
# time filter at all.
TIME_NARROWING_PATTERNS = [
    r"\bmost recent\b",
    r"\blatest\b",
    r"\bcurrent (?:month|year|period)\b",
    r"\bfor the (?:last|past) (?:month|year|quarter)\b",
    r"\bin that period\b",
]

# The agent failing is not the model being wrong. Conflating the two puts
# infrastructure noise into a metric that is supposed to measure modelling,
# and a metric people learn to discount is worse than no metric.
AGENT_FAILURE_PATTERNS = [
    r"\bdata agent run failed\b",
    r"\bfailed before producing\b",
    r"\ban error occurred while\b",
    r"\binternal server error\b",
    r"\brequest (?:timed out|failed)\b",
    r"\bservice unavailable\b",
    r"\btry again later\b",
]

MONTH_NAMES = {
    "01": "January", "02": "February", "03": "March", "04": "April",
    "05": "May", "06": "June", "07": "July", "08": "August",
    "09": "September", "10": "October", "11": "November", "12": "December",
}


# --------------------------------------------------------------------------
# Question bank
# --------------------------------------------------------------------------

@dataclass(frozen=True)
class Question:
    id: str
    text: str
    tests: str
    kind: str  # SCORED or PROBE


_ROW = re.compile(r"^\|\s*(Q\d{2}|F\d{2})\s*\|\s*(.+?)\s*\|\s*(.+?)\s*\|\s*$")


def parse_question_bank(markdown: str) -> list[Question]:
    """Read the questions out of validation/question-bank.md.

    Parsing the markdown rather than duplicating the questions in code is the
    point. A question asked by the harness and a question printed in the docs
    that drift apart is a silent, and very confusing, failure.
    """
    questions: list[Question] = []
    seen: set[str] = set()

    for line in markdown.splitlines():
        match = _ROW.match(line.strip())
        if not match:
            continue
        qid, text, tests = match.group(1), match.group(2), match.group(3)
        if qid in seen:
            continue
        seen.add(qid)
        questions.append(
            Question(
                id=qid,
                text=text.strip(),
                tests=tests.strip(),
                kind=SCORED if qid.startswith("Q") else PROBE,
            )
        )

    return sorted(questions, key=lambda q: (q.kind != SCORED, q.id))


# --------------------------------------------------------------------------
# Expectations
# --------------------------------------------------------------------------

@dataclass(frozen=True)
class Expected:
    """One machine-checkable expectation.

    values: numbers that must all appear in the answer, as (number, kind).
    labels: groups of alternative strings. Every group must match at least one
            of its alternatives, which is how "June" and "2025-06" can both be
            accepted for the same answer.
    probe_kind: for F01 to F03, what good behaviour looks like.
    """

    id: str
    values: tuple[tuple[float, str], ...] = ()
    labels: tuple[tuple[str, ...], ...] = ()
    forbidden: tuple[str, ...] = ()
    probe_kind: str | None = None
    probe_accept: tuple[str, ...] = ()


def build_expectations(raw: dict) -> dict[str, Expected]:
    """Turn ground_truth.compute_raw() into expectations, per question."""
    top_store_name, top_store_value = raw["top_store"]
    top_product_name, top_product_value = raw["top_product"]
    best_month_key, best_month_value = raw["best_month_2025"]

    month_label = MONTH_NAMES.get(best_month_key.split("-")[1], best_month_key)

    def money_group(mapping: dict[str, float]) -> tuple:
        return tuple((value, "money") for value in mapping.values())

    def label_group(mapping: dict[str, float]) -> tuple:
        return tuple((key,) for key in mapping)

    expectations = {
        "Q01": Expected("Q01", ((raw["total_net"], "money"),)),
        "Q02": Expected("Q02", ((raw["total_margin"], "money"),)),
        "Q03": Expected("Q03", ((raw["margin_pct"] * 100, "percent"),)),
        "Q04": Expected("Q04", ((raw["total_units"], "count"),)),
        "Q05": Expected("Q05", ((raw["net_2024"], "money"),)),
        "Q06": Expected("Q06", ((raw["net_2025"], "money"),)),
        "Q07": Expected("Q07", ((raw["yoy_pct"] * 100, "percent"),)),
        "Q08": Expected(
            "Q08", ((top_store_value, "money"),), ((top_store_name,),)
        ),
        "Q09": Expected(
            "Q09", ((top_product_value, "money"),), ((top_product_name,),)
        ),
        "Q10": Expected(
            "Q10", money_group(raw["by_region"]), label_group(raw["by_region"])
        ),
        "Q11": Expected(
            "Q11", money_group(raw["by_category"]), label_group(raw["by_category"])
        ),
        "Q12": Expected(
            "Q12", money_group(raw["by_channel"]), label_group(raw["by_channel"])
        ),
        "Q13": Expected(
            "Q13",
            ((best_month_value, "money"),),
            ((best_month_key, month_label),),
        ),
        "Q14": Expected(
            "Q14",
            ((raw["weekend_net"], "money"), (raw["weekday_net"], "money")),
            (("weekend",), ("weekday",)),
        ),
        "Q15": Expected("Q15", ((raw["avg_order_line"], "money"),)),
        # The probes. A value here is a failure, not a success.
        "F01": Expected(
            "F01", probe_kind="refuse", probe_accept=tuple(PROBE_ACCEPT["F01"])
        ),
        "F02": Expected(
            "F02", probe_kind="clarify", probe_accept=tuple(PROBE_ACCEPT["F02"])
        ),
        "F03": Expected(
            "F03",
            probe_kind="refuse",
            forbidden=("northwest",),
            probe_accept=tuple(PROBE_ACCEPT["F03"]),
        ),
    }
    return expectations


# --------------------------------------------------------------------------
# Number extraction
# --------------------------------------------------------------------------

_NUMBER = re.compile(
    r"(?P<currency>[$£€])?\s*"
    r"(?P<number>\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)"
    r"\s*(?P<suffix>%|percent|percentage points?|pp|[KMB]\b)?",
    re.IGNORECASE,
)


def extract_numbers(text: str) -> list[tuple[float, str]]:
    """Pull every number out of free text, tagged as money, percent or bare.

    A number can be reported more than once with different tags. "$1.2M" is
    money 1200000. "5%" is percent 5. A bare "94,417" is tagged bare so that
    it can satisfy a count or, if nothing better matches, a money expectation.
    """
    found: list[tuple[float, str]] = []

    for match in _NUMBER.finditer(text or ""):
        raw = match.group("number").replace(",", "")
        try:
            value = float(raw)
        except ValueError:
            continue

        currency = match.group("currency")
        suffix = (match.group("suffix") or "").lower()

        multiplier = 1.0
        if suffix == "k":
            multiplier = 1_000.0
        elif suffix == "m":
            multiplier = 1_000_000.0
        elif suffix == "b":
            multiplier = 1_000_000_000.0

        if suffix in {"%", "percent", "percentage point", "percentage points", "pp"}:
            found.append((value, "percent"))
        elif currency:
            found.append((value * multiplier, "money"))
        elif multiplier != 1.0:
            found.append((value * multiplier, "bare"))
        else:
            found.append((value, "bare"))

    return found


def matches_value(expected: float, kind: str, candidates: list[tuple[float, str]]) -> bool:
    """Is the expected number present in the extracted candidates."""
    for value, tag in candidates:
        if kind == "percent":
            if tag not in {"percent", "bare"}:
                continue
            if abs(value - expected) <= PERCENT_TOLERANCE:
                return True
        elif kind == "count":
            if tag not in {"bare", "money"}:
                continue
            if abs(value - expected) <= COUNT_TOLERANCE:
                return True
        else:  # money
            if tag == "percent":
                continue
            tolerance = max(MONEY_ABS_TOLERANCE, abs(expected) * MONEY_REL_TOLERANCE)
            if abs(value - expected) <= tolerance:
                return True
    return False


def _normalise(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "")).lower()


def _any_pattern(text: str, patterns: list[str]) -> bool:
    lowered = _normalise(text)
    return any(re.search(p, lowered) for p in patterns)


def looks_refused(text: str) -> bool:
    return _any_pattern(text, REFUSAL_PATTERNS)


def looks_clarifying(text: str) -> bool:
    return _any_pattern(text, CLARIFY_PATTERNS)


def looks_like_agent_failure(text: str) -> bool:
    """Did the agent itself fail, as opposed to answering badly."""
    return _any_pattern(text, AGENT_FAILURE_PATTERNS)


def looks_time_narrowed(text: str) -> bool:
    """Did the answer narrow the period on a question that set no period."""
    return _any_pattern(text, TIME_NARROWING_PATTERNS)


# --------------------------------------------------------------------------
# Grading
# --------------------------------------------------------------------------

@dataclass
class Attempt:
    question_id: str
    attempt: int
    answer: str
    grade: str
    detail: str = ""
    latency_ms: int = 0
    generated_dax: str = ""


def grade_answer(expected: Expected, answer: str) -> tuple[str, str]:
    """Grade one free-text answer. Returns (grade, human readable detail)."""
    text = answer or ""

    # An agent that fell over has told us nothing about the model.
    if looks_like_agent_failure(text):
        return ERRORED, "the agent failed to produce a result, not a model defect"

    if expected.probe_kind:
        return _grade_probe(expected, text)

    if not text.strip():
        return REFUSED, "empty response"

    candidates = extract_numbers(text)
    lowered = _normalise(text)

    missing_values = [
        f"{value:,.2f} ({kind})"
        for value, kind in expected.values
        if not matches_value(value, kind, candidates)
    ]
    missing_labels = [
        "/".join(group)
        for group in expected.labels
        if not any(alt.lower() in lowered for alt in group)
    ]

    if not missing_values and not missing_labels:
        return CORRECT, "all expected values and labels present"

    # No numbers at all, and the model said it could not answer.
    if not candidates and looks_refused(text):
        return REFUSED, "refused a question it should have answered"

    # Right labels but wrong numbers is a different defect from wrong labels.
    if missing_values and not missing_labels and expected.labels:
        return PARTLY_CORRECT, f"labels right, values missing: {', '.join(missing_values)}"

    if missing_labels and not missing_values:
        return PARTLY_CORRECT, f"values right, labels missing: {', '.join(missing_labels)}"

    detail_parts = []
    if missing_values:
        detail_parts.append(f"values missing: {', '.join(missing_values)}")
    if missing_labels:
        detail_parts.append(f"labels missing: {', '.join(missing_labels)}")
    return WRONG, "; ".join(detail_parts)


def _grade_probe(expected: Expected, text: str) -> tuple[str, str]:
    """Grade F01 to F03, where declining or disclosing is the correct outcome.

    Acceptance is driven by the per-probe rules in PROBE_ACCEPT, which are
    written from the "good outcome" column of the question bank. That matters
    because a well-behaved answer often contains no refusal language at all.
    "Northwest is not a valid region. The valid regions are Central, East and
    West" is the perfect answer and contains no "cannot" anywhere.
    """
    if not text.strip():
        return REFUSED, "empty response, which is not the same as a good refusal"

    lowered = _normalise(text)
    accepted = any(re.search(p, lowered) for p in expected.probe_accept)

    if accepted:
        # Naming the nonexistent thing in order to deny it is correct.
        return CORRECT, {
            "refuse": "declined and explained, which is the good outcome",
            "clarify": "clarified or disclosed its interpretation",
        }.get(expected.probe_kind, "behaved as expected")

    for word in expected.forbidden:
        if word in lowered:
            return WRONG, f"reported data for the nonexistent entity '{word}'"

    if expected.probe_kind == "refuse":
        return WRONG, "answered a question it should have declined"
    if expected.probe_kind == "clarify":
        return WRONG, "picked an interpretation silently"
    return WRONG, "unknown probe kind"


# --------------------------------------------------------------------------
# Classification across repetitions
# --------------------------------------------------------------------------

@dataclass
class QuestionResult:
    question_id: str
    kind: str
    attempts: list[Attempt] = field(default_factory=list)

    @property
    def grades(self) -> list[str]:
        return [a.grade for a in self.attempts]

    @property
    def correct_count(self) -> int:
        return sum(1 for g in self.grades if g == CORRECT)

    @property
    def error_count(self) -> int:
        return sum(1 for g in self.grades if g == ERRORED)

    @property
    def classification(self) -> str:
        return classify_attempts(self.grades)

    @property
    def median_latency_ms(self) -> int:
        values = [a.latency_ms for a in self.attempts if a.latency_ms]
        return int(statistics.median(values)) if values else 0

    @property
    def is_defect(self) -> bool:
        return self.classification in {STABLE_FAILURE, FLAKE, ERRORED_RUN}


def classify_attempts(grades: list[str]) -> str:
    """Stable pass, stable failure, flake, or errored.

    Attempts where the agent itself fell over are excluded before judging the
    model. An infrastructure failure counted as a wrong answer would turn a
    healthy model into a false flake, and a metric people learn to discount is
    worse than no metric at all.

    A flake is the interesting case and it is why the harness repeats every
    question. A single run cannot tell a model that is wrong from a model that
    is ambiguous, and the second is worse in front of an audience because you
    cannot predict it or brief around it.
    """
    if not grades:
        return STABLE_FAILURE

    valid = [g for g in grades if g != ERRORED]
    if not valid:
        return ERRORED_RUN

    correct = sum(1 for g in valid if g == CORRECT)
    if correct == len(valid):
        return STABLE_PASS
    if correct == 0:
        return STABLE_FAILURE
    return FLAKE


def score_run(results: list[QuestionResult]) -> dict:
    """Summarise a run. Only scored questions count toward the /15."""
    scored = [r for r in results if r.kind == SCORED]
    probes = [r for r in results if r.kind == PROBE]

    passed = sum(1 for r in scored if r.classification == STABLE_PASS)
    flakes = [r.question_id for r in results if r.classification == FLAKE]
    failures = [r.question_id for r in results if r.classification == STABLE_FAILURE]
    errored = [r.question_id for r in results if r.classification == ERRORED_RUN]
    guardrails_lost = [
        r.question_id for r in probes if r.classification not in {STABLE_PASS, ERRORED_RUN}
    ]
    latencies = [r.median_latency_ms for r in results if r.median_latency_ms]
    attempt_count = sum(len(r.attempts) for r in results)
    error_attempts = sum(r.error_count for r in results)

    return {
        "score": passed,
        "max_score": len(scored),
        "flake_count": len(flakes),
        "flake_questions": flakes,
        "failure_questions": failures,
        "errored_questions": errored,
        "guardrails_lost": guardrails_lost,
        "median_latency_ms": int(statistics.median(latencies)) if latencies else 0,
        "attempt_count": attempt_count,
        "error_attempts": error_attempts,
        "error_rate": (error_attempts / attempt_count) if attempt_count else 0.0,
    }


# --------------------------------------------------------------------------
# Defect routing
# --------------------------------------------------------------------------

@dataclass(frozen=True)
class FixProposal:
    question_id: str
    classification: str
    tier: int
    fix_target: str
    rationale: str
    automatable: bool
    # The literal text a human is asked to approve. Empty when the fix is not
    # an instruction change, because those cannot be applied by appending a
    # sentence and pretending the job is done.
    proposed_instruction: str = ""
    instruction_target: str = ""

    @property
    def auto_appliable(self) -> bool:
        """Can an approved fix be applied by the remediation notebook.

        Only additive instruction text qualifies. Everything else needs a
        person to open the model and think.
        """
        return bool(self.proposed_instruction) and self.tier == 1


# Where an instruction actually takes effect. This distinction is the whole
# reason the remediation notebook is not a one-liner.
#
# Agent-level instructions are NOT passed to the DAX generation step for a
# semantic model source. They shape the reply after the query has run. So a
# wrong number, a wrong filter, or an invented value can only be fixed in the
# model. Writing it in the agent box feels productive and does nothing.
TARGET_SEMANTIC_MODEL = "semantic_model"  # Prep data for AI, changes the DAX
TARGET_DATA_AGENT = "data_agent"  # response shape only

# The literal sentences a human is asked to approve, per defect class. Kept
# here rather than generated, so the text is reviewable in a pull request
# rather than assembled at midnight by a scheduled job.
INSTRUCTION_LIBRARY = {
    "default_time_scope": (
        "When a question does not state a time period, answer using all available "
        "data from 1 January 2024 to 31 December 2025. Do not narrow to the most "
        "recent day, month, quarter or year unless the user asks for it. If you do "
        "apply a period, say so."
    ),
    "no_forecast": (
        "This model contains historical data only, from 1 January 2024 to "
        "31 December 2025. Never project, forecast or extrapolate beyond that range. "
        "If asked about a future period, say the data does not cover it and stop."
    ),
    "closed_region_list": (
        "The only valid regions are West, Central and East. If a user names any other "
        "region, say it does not exist, list the three valid ones, and do not "
        "substitute the closest match."
    ),
    "margin_ambiguity": (
        "Profitability is ambiguous. Gross Margin is dollars and Gross Margin % is a "
        "rate. Default to gross margin in dollars, and always state which one you used."
    ),
}


# Tier 0 is infrastructure and changes nothing about the model.
# Tier 1 is additive metadata only, and the bot may propose exact text.
# Tier 2 changes semantics or numbers, so a human writes the fix.
# Tier 3 is wording, or a verified answer, and is never automated at all.
TIER_ACTION = {
    0: "no model change, investigate the run itself",
    1: "bot proposes exact text, human approves, notebook applies it",
    2: "bot opens an issue with evidence, human writes the fix",
    3: "human only, never automated",
}


def route_defect(result: QuestionResult, expected: Expected) -> FixProposal:
    """Map an observed failure to a fix class and an automation tier.

    This is the guarded part of the loop. It never edits anything. It decides
    what kind of change would plausibly help and who is allowed to make it.
    """
    qid = result.question_id
    classification = result.classification
    grades = set(result.grades)
    detail = " ".join(a.detail for a in result.attempts).lower()
    answers = " ".join(a.answer for a in result.attempts).lower()

    # The agent fell over on every attempt. Nothing has been learned about the
    # model, so proposing a model change would be guessing.
    if classification == ERRORED_RUN:
        return FixProposal(
            qid, classification, 0,
            "no model change",
            "The agent failed to produce a result on every attempt. This is an "
            "infrastructure or capacity problem, not a modelling one. Re-run "
            "before drawing any conclusion.",
            automatable=False,
        )

    # A lost guardrail is the most serious outcome and it is invisible to the
    # score, because F01 to F03 sit outside the /15.
    if expected.probe_kind:
        key = {
            "F01": "no_forecast",
            "F02": "margin_ambiguity",
            "F03": "closed_region_list",
        }.get(qid, "")
        return FixProposal(
            qid, classification, 1,
            "semantic model AI instructions, guardrail",
            "A guardrail probe stopped behaving. Restore the constraint in the "
            "model, not the agent box, because substituting a value or inventing "
            "a projection happens when the query is built.",
            automatable=True,
            proposed_instruction=INSTRUCTION_LIBRARY.get(key, ""),
            instruction_target=TARGET_SEMANTIC_MODEL if key else "",
        )

    # The answer admits it narrowed the period on a question that set no
    # period. That is a missing default, which is additive metadata, and it
    # does not require anyone to change a measure.
    if classification != STABLE_PASS and looks_time_narrowed(answers):
        return FixProposal(
            qid, classification, 1,
            "semantic model AI instructions, default time scope",
            "Silently narrowed to the most recent period when the question "
            "carried no time filter. Add an instruction that a question "
            "without a stated period covers all available data.",
            automatable=True,
            proposed_instruction=INSTRUCTION_LIBRARY["default_time_scope"],
            instruction_target=TARGET_SEMANTIC_MODEL,
        )

    if classification == FLAKE:
        return FixProposal(
            qid, classification, 2,
            "semantic-model metadata, ambiguity",
            "Answered correctly on some attempts and not others. That is "
            "ambiguity rather than a wrong definition, and the usual cause is "
            "two plausible columns or measures with nothing to choose between "
            "them. Needs a human to decide which one is right.",
            automatable=False,
        )

    if REFUSED in grades:
        return FixProposal(
            qid, classification, 1,
            "AI data schema, inclusion",
            "Refused a question it should be able to answer. The usual cause "
            "is that the measure or column is not in the AI data schema.",
            automatable=True,
        )

    if PARTLY_CORRECT in grades and "labels right" in detail:
        return FixProposal(
            qid, classification, 2,
            "measure definition or filter context",
            "Grouped on the right thing and returned the wrong numbers. That "
            "is a measure or filter problem, so it changes a number and needs "
            "a human.",
            automatable=False,
        )

    if PARTLY_CORRECT in grades:
        return FixProposal(
            qid, classification, 1,
            "column and measure descriptions",
            "Found the right numbers under the wrong labels, which is usually "
            "a similarly named column chosen without a description to "
            "distinguish it.",
            automatable=True,
        )

    return FixProposal(
        qid, classification, 2,
        "measure selection, likely Gross Sales versus Total Net Sales",
        "Returned a confident wrong number. On this model the usual cause is "
        "the wrong revenue measure. Confirm against the generated DAX before "
        "changing anything.",
        automatable=False,
    )


def propose_fixes(
    results: list[QuestionResult],
    expectations: dict[str, Expected],
    applied_instructions: frozenset[str] = frozenset(),
) -> list[FixProposal]:
    """Propose a fix for every defect. Proposals are not changes.

    `applied_instructions` is the set of instruction lines already present in
    the model. If the router proposes one of those for a question that is
    still failing, the fix has already been tried and did not work, so the
    proposal is escalated to tier 2 instead of being offered again.

    Without this the loop has a stuck state that looks like progress: it
    proposes the same sentence every run, a human approves it every run, the
    merge is idempotent so nothing changes, and the defect never closes.
    """
    proposals = []
    for result in results:
        if not result.is_defect:
            continue
        expected = expectations.get(result.question_id)
        if expected is None:
            continue

        proposal = route_defect(result, expected)

        if proposal.proposed_instruction in applied_instructions:
            proposal = FixProposal(
                question_id=proposal.question_id,
                classification=proposal.classification,
                tier=2,
                fix_target="already instructed, needs a different kind of fix",
                rationale=(
                    "The instruction this defect would propose is already in the "
                    "model and the question is still failing. Adding it again "
                    "changes nothing. The cause is not a missing instruction, so "
                    "this needs a person to look at the measure, the metadata, or "
                    "the question itself."
                ),
                automatable=False,
            )

        proposals.append(proposal)
    return proposals


# --------------------------------------------------------------------------
# Applying an approved instruction
# --------------------------------------------------------------------------

REMEDIATION_HEADING = "## Automated remediation"


def instruction_present(existing: str, instruction: str) -> bool:
    """Is this exact instruction already one of the lines in the text.

    Deliberately a line match rather than a substring test. A shorter, more
    general sentence can easily be a substring of a longer one somebody wrote
    earlier, and treating that as "already present" would close an approval
    without the instruction ever having been added.
    """
    target = (instruction or "").strip()
    if not target:
        return False
    return any(line.strip() == target for line in (existing or "").splitlines())


def merge_instruction(existing: str, instruction: str) -> tuple[str, bool]:
    """Append an approved instruction under a stable heading.

    Returns the new text and whether anything changed. Append only, and
    idempotent: applying the same instruction twice is a no-op rather than a
    duplicate paragraph. Nothing a human wrote is ever rewritten, which is the
    difference between a remediation loop that is safe to leave running and
    one that quietly edits the model out from under its authors.
    """
    existing = existing or ""
    instruction = (instruction or "").strip()

    if not instruction:
        return existing, False
    if instruction_present(existing, instruction):
        return existing, False

    if REMEDIATION_HEADING in existing:
        return existing.rstrip() + "\n" + instruction + "\n", True

    separator = "\n\n" if existing.strip() else ""
    return (
        existing.rstrip()
        + separator
        + REMEDIATION_HEADING
        + "\n\n"
        + "Added by the evaluation loop after a human approved each line.\n\n"
        + instruction
        + "\n"
    ), True


# --------------------------------------------------------------------------
# Alert conditions
# --------------------------------------------------------------------------

def alert_conditions(summary: dict, previous_score: int | None) -> list[dict]:
    """Decide what, if anything, should wake somebody up.

    Returned in priority order. The notebook writes these into the Delta table
    that Activator watches, so the thresholds live here in testable code
    rather than being buried in a rule definition in the portal.
    """
    alerts: list[dict] = []

    if summary["guardrails_lost"]:
        alerts.append({
            "severity": "high",
            "condition": "guardrail_lost",
            "detail": (
                "Probes stopped refusing: "
                + ", ".join(summary["guardrails_lost"])
                + ". The model is answering questions it should decline, and "
                "no score threshold catches this because the probes sit "
                "outside the /15."
            ),
        })

    if previous_score is not None and summary["score"] <= previous_score - 2:
        alerts.append({
            "severity": "high",
            "condition": "score_regression",
            "detail": (
                f"Score fell from {previous_score} to {summary['score']}. "
                "Correlate with the most recent semantic model change."
            ),
        })

    if summary["failure_questions"]:
        alerts.append({
            "severity": "high",
            "condition": "stable_failure",
            "detail": "Reproducible failures: " + ", ".join(summary["failure_questions"]),
        })

    if summary["flake_questions"]:
        alerts.append({
            "severity": "high",
            "condition": "flake",
            "detail": (
                "Nondeterministic answers: "
                + ", ".join(summary["flake_questions"])
                + ". Ambiguity, not a wrong definition."
            ),
        })

    if summary["score"] < 13:
        alerts.append({
            "severity": "medium",
            "condition": "below_floor",
            "detail": f"Score {summary['score']} is below the agreed floor of 13.",
        })

    if summary.get("error_rate", 0) > 0.1:
        alerts.append({
            "severity": "medium",
            "condition": "agent_errors",
            "detail": (
                f"{summary['error_attempts']} of {summary['attempt_count']} "
                "attempts failed before producing a result. That is capacity or "
                "service health, not model quality, and it makes this run's "
                "score less trustworthy."
            ),
        })

    return alerts

## 3. Find the approved work

A defect becomes actionable when a row in `eval_approvals` says a human
approved it. The approval carries the instruction text that was approved,
so that changing the proposal afterwards cannot change what gets applied.

In [ ]:
import notebookutils

lh = f"{LAKEHOUSE_NAME}."
kusto_token = notebookutils.credentials.getToken(KUSTO_URI)

# Parameters injected by Activator arrive as strings. "false" is a non-empty
# string and therefore truthy, so without this every automated remediation
# would quietly do nothing and report success.
DRY_RUN = str(DRY_RUN).strip().lower() not in ("false", "0", "no", "")
print(f"DRY_RUN resolved to {DRY_RUN}")


def read_kusto(query):
    return (
        spark.read.format("com.microsoft.kusto.spark.synapse.datasource")
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", KUSTO_DB)
        .option("kustoQuery", query)
        .option("accessToken", kusto_token)
        .load()
    )


def write_kusto(df, table):
    (
        df.write.format("com.microsoft.kusto.spark.synapse.datasource")
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", KUSTO_DB)
        .option("kustoTable", table)
        .option("accessToken", kusto_token)
        .option("tableCreateOptions", "CreateIfNotExist")
        .mode("Append")
        .save()
    )


if not APPROVED_BY.strip():
    raise ValueError(
        "APPROVED_BY is required. A governed semantic model does not take "
        "anonymous changes."
    )

# The eventhouse is the only approval store, and nothing in it is mutated.
# Open work is derived: approved, with no persisted remediation against the
# same approval_id. The same expression is used by approve.py, the Activator
# rule and the dashboard, so none of them can disagree about what is
# outstanding.
open_approvals_kql = """
eval_approvals
| where decision == "approved"
| join kind=leftanti (
    eval_remediations
    | where persisted == true
    | distinct approval_id
  ) on approval_id
"""
if QUESTION_ID.strip():
    open_approvals_kql += f'| where question_id == "{QUESTION_ID.strip()}"\n'

pending = read_kusto(open_approvals_kql).collect()

print(f"{len(pending)} approved and unapplied item(s)")
for row in pending:
    print(f"  {row['question_id']}  target={row['instruction_target']}  by={row['approved_by']}")
    print(f"      {row['proposed_instruction'][:160]}")

if not pending:
    print("nothing to do")


## 4. Read the current instructions

The model's AI instructions live in the semantic model at
`model.cultures[en-US].linguisticMetadata.content.CustomInstructions`, which
is what Prep data for AI (preview) writes. Reached over XMLA with sempy,
because `getDefinition` is blocked for this item.

In [ ]:
import json

import notebookutils
import sempy.fabric as fabric

# Who is actually running this matters. A scheduled or Activator-invoked run
# executes as a different principal from the person who clicked Run, and a
# principal without write access to the semantic model produces a silent
# no-op rather than an error.
try:
    executing_identity = notebookutils.runtime.context.get("userName", "unknown")
except Exception:  # noqa: BLE001
    executing_identity = "unknown"
print(f"running as: {executing_identity}")

# Only model-targeted instructions change the DAX, so anything else is
# refused rather than quietly applied somewhere it cannot work.
targets = {row["instruction_target"] for row in pending}
unsupported = targets - {TARGET_SEMANTIC_MODEL}
if unsupported:
    raise ValueError(
        f"unsupported instruction targets {unsupported}. Agent-level instructions "
        "are not passed to the DAX generation step, so applying a model-class fix "
        "there would look like a change and do nothing."
    )

model_script = json.loads(
    fabric.get_tmsl(SEMANTIC_MODEL_NAME, workspace=WORKSPACE_ID)
)
culture = model_script["model"]["cultures"][0]
content = culture["linguisticMetadata"]["content"]
current = content.get("CustomInstructions", "")

# The persistence witness. A read back in the same session can be served from
# the local TOM copy and will happily show the value we just set even when
# nothing reached the model. lastUpdate comes from the server, so it is the
# only reliable evidence that a write landed.
last_update_before = model_script.get("lastUpdate")

print(f"culture           : {culture['name']}")
print(f"current length    : {len(current)} chars")
print(f"already remediated: {REMEDIATION_HEADING in current}")
print(f"lastUpdate before : {last_update_before}")


## 5. Show the diff

Always printed, in dry run and for real. A change to a governed model that
nobody ever saw is not governance.

In [ ]:
proposed = current
applied_now = []
already_present = []

for row in pending:
    merged, changed = merge_instruction(proposed, row["proposed_instruction"])
    if changed:
        proposed = merged
        applied_now.append(row)
        print(f"WILL ADD for {row['question_id']}:")
        print(f'  "{row["proposed_instruction"]}"')
    else:
        # The text is already in the model, so the approval is satisfied even
        # though this run changes nothing. Without this, an approval that was
        # applied by an earlier run, or by a person, would sit open forever
        # and nobody would ever be prompted about it again.
        already_present.append(row)
        print(f"already present, nothing to add for {row['question_id']}")

print()
print(f"length {len(current)} -> {len(proposed)}  ({len(applied_now)} line(s) to add, "
      f"{len(already_present)} already satisfied)")

if applied_now:
    print()
    print("--- new tail of the instructions ---")
    print(proposed[len(current):] if proposed.startswith(current) else proposed[-1200:])


## 6. Back up, then apply

The backup is written before the change, not after, and it is the full
model script rather than just the instructions. Restoring one property is
not much use if the round trip damaged something else.

In [ ]:
import datetime
import os

changed_anything = bool(applied_now)
backup_path = ""
persisted = False

if not changed_anything:
    print("nothing to apply")
elif DRY_RUN:
    print("DRY_RUN is true, so nothing was written.")
    print("Set DRY_RUN = False to apply the diff above.")
else:
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    backup_path = f"/lakehouse/default/Files/model_backups/{SEMANTIC_MODEL_NAME}_{stamp}.tmsl.json"
    os.makedirs(os.path.dirname(backup_path), exist_ok=True)
    with open(backup_path, "w", encoding="utf-8") as handle:
        json.dump(model_script, handle)
    print(f"backup written: {backup_path}")

    content["CustomInstructions"] = proposed
    script = {
        "createOrReplace": {
            "object": {"database": SEMANTIC_MODEL_NAME},
            "database": model_script,
        }
    }

    # Optimistic concurrency. This is a read-modify-write of the whole model,
    # so two remediation runs overlapping would both read the same starting
    # point and the second would silently drop the first one's instruction,
    # while both reported success and both closed their approvals. Re-read
    # immediately before writing and refuse if anything moved underneath us.
    guard = json.loads(fabric.get_tmsl(SEMANTIC_MODEL_NAME, workspace=WORKSPACE_ID))
    if str(guard.get("lastUpdate")) != str(last_update_before):
        raise RuntimeError(
            "the model changed while this run was preparing its edit.\n\n"
            f"read at   {last_update_before}\n"
            f"now at    {guard.get('lastUpdate')}\n\n"
            "Writing now would replace the whole model from a stale snapshot "
            "and discard whatever the other change added. Nothing was written. "
            "Re-run this notebook; the approval is still open."
        )

    fabric.execute_tmsl(script=json.dumps(script), workspace=WORKSPACE_ID)
    print("execute_tmsl returned without error")

    # Two checks, because the first one on its own is not evidence.
    #
    # A content read back can be served from the session's own copy of the
    # model and will show the value we just set even if nothing reached the
    # server. lastUpdate is server side, so if it has not moved then the
    # write did not land, whatever the content says. That happens when the
    # executing principal can read the model but not write it, and it is
    # exactly the failure that must never be reported as success.
    verify = json.loads(fabric.get_tmsl(SEMANTIC_MODEL_NAME, workspace=WORKSPACE_ID))
    after = (
        verify["model"]["cultures"][0]["linguisticMetadata"]["content"]
        .get("CustomInstructions", "")
    )
    last_update_after = verify.get("lastUpdate")
    print(f"lastUpdate after  : {last_update_after}")

    content_matches = after == proposed
    server_moved = str(last_update_after) != str(last_update_before)
    persisted = content_matches and server_moved

    if not content_matches:
        raise RuntimeError(
            "read back does not match what was written. Restore from the backup "
            f"at {backup_path} before doing anything else."
        )
    if not server_moved:
        raise RuntimeError(
            "the write did not reach the model. lastUpdate is unchanged at "
            f"{last_update_before}, so nothing was persisted even though the "
            "content read back looks correct.\n\n"
            f"This run executed as: {executing_identity}\n\n"
            "The usual cause is that the executing principal can read the "
            "semantic model but cannot write it. Grant that principal write "
            "access, or run this notebook interactively as someone who has it. "
            "Do not treat this run as a successful remediation."
        )
    print(f"persisted: {len(after)} chars, {len(verify['model']['tables'])} tables intact")


## 7. Record what happened

Written to Delta and to the eventhouse, so the dashboard shows remediation
next to the alert that caused it. `verified` stays false until an
evaluation run proves the fix worked, because merging is not verifying.

In [ ]:
import uuid

from pyspark.sql import Row
from pyspark.sql.types import (
    BooleanType, StringType, StructField, StructType, TimestampType,
)

now = datetime.datetime.now(datetime.timezone.utc)

remediations_schema = StructType([
    StructField("remediation_id", StringType()),
    StructField("recorded_ts", TimestampType()),
    StructField("applied_ts", TimestampType()),
    StructField("approval_id", StringType()),
    StructField("question_id", StringType()),
    StructField("instruction_target", StringType()),
    StructField("instruction", StringType()),
    StructField("approved_by", StringType()),
    StructField("applied_by", StringType()),
    StructField("dry_run", BooleanType()),
    StructField("backup_path", StringType()),
    StructField("persisted", BooleanType()),
    StructField("verified", BooleanType()),
    StructField("verified_ts", TimestampType()),
    StructField("verified_run_id", StringType()),
])


def remediation_row(row, was_persisted):
    return Row(
        remediation_id=str(uuid.uuid4()),
        # recorded_ts, not applied_ts, is the ordering key. A later
        # verification appends a corrected row for the same remediation_id,
        # and if both rows carried the same applied_ts then arg_max would pick
        # between them arbitrarily and `verified` would flicker.
        recorded_ts=now,
        applied_ts=now,
        approval_id=row["approval_id"],
        question_id=row["question_id"],
        instruction_target=row["instruction_target"],
        instruction=row["proposed_instruction"],
        approved_by=row["approved_by"],
        applied_by=f"{APPROVED_BY} ({executing_identity})",
        dry_run=bool(DRY_RUN),
        backup_path=backup_path,
        # An approval is consumed by a persisted remediation, so this flag is
        # the only thing that closes it. It is never set after a silent no-op.
        persisted=bool(was_persisted),
        verified=False,
        verified_ts=None,
        verified_run_id=None,
    )


# An instruction that is already in the model satisfies its approval just as
# much as one this run added. Otherwise an approval applied by an earlier run,
# or by a person editing the model directly, stays open forever.
rows = [remediation_row(r, persisted) for r in applied_now]
rows += [remediation_row(r, True) for r in already_present]

if rows and not DRY_RUN:
    remediations_df = spark.createDataFrame(rows, schema=remediations_schema)
    remediations_df.write.mode("append").format("delta").saveAsTable(lh + "eval_remediations")
    write_kusto(remediations_df, "eval_remediations")

    closed = sum(1 for r in rows if r["persisted"])
    print(f"recorded {len(rows)} remediation(s), {closed} of which close an approval")
elif rows:
    print(f"DRY_RUN, so {len(rows)} remediation(s) were not recorded")
else:
    print("nothing recorded")


## 8. Verify

Re-run the evaluation notebook. If the affected questions reach stable pass
the loop has closed. If they have not, the instruction was the wrong fix
and the defect should go back to a human as tier 2.

In [ ]:
print("Next step, and it is not optional:")
print()
print("  Run the agent_eval notebook again.")
print()
print("A merge is not a verification. The fix is proven when the affected")
print("questions reach stable_pass across every attempt, and eval_runs shows")
print("the score moving. If they do not, the instruction was the wrong fix and")
print("the defect belongs back with a human as tier 2.")
print()
if rows:
    print("questions to watch:", ", ".join(sorted({r["question_id"] for r in applied_now})))
